<a href="https://colab.research.google.com/github/Asuskf/GECTOR-Spanish-Medical-Correction/blob/main/lab_MLOps_esp_fixed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## MLOps ♾️
## Etapa 1: DISEÑO


### Fase 1 Ingeniería de Requisitos

**🏥 Caso de Uso: Auditoría Hospital Y**
El Hospital Y necesita evaluar los procedimientos realizados por su personal médico (doctores y enfermeras). Durante la extracción de datos, se detectó una alta incidencia de faltas de ortografía y errores tipográficos en las notas clínicas, causados por la premura al escribir durante las atenciones.

**🎯 Objetivo Principal:** Limpiar y corregir las faltas ortográficas en los registros de texto para viabilizar la evaluación de los procedimientos.

### 👥 Paso 2: Priorización del Caso de Uso: Audiencia y Meta Analítica

Para que este esfuerzo de MLOps tenga sentido, el diseño debe estar anclado a una necesidad de negocio clara.

* **Audiencia Final:** Directivos del Hospital Y (Perfil Administrativo / Toma de decisiones).
* **El verdadero objetivo (El "Por qué"):** Los directivos no necesitan leer las historias clínicas individuales. Necesitan analizar la **macrotendencia operativa** del hospital para asignar recursos adecuadamente (ej. ¿Estamos derivando más pacientes de los que operamos? ¿Cuál es el volumen real de procedimientos ambulatorios?).
* **El Bloqueo Actual:** Si el personal escribe "cirujia", "cirg", o "sirugia" por la premura, los sistemas de Business Intelligence (BI) no logran agrupar estos términos.
* **Justificación del Modelo:** El modelo de corrección ortográfica actúa como el habilitador principal (DataOps) para que los *dashboards* gerenciales reflejen la realidad operativa del hospital.**texto en negrita**

### 🧩 Paso 3: Desglosar el Problema (Hacerlo Abordable)

Como vimos en el diseño, intentar pasar de "texto médico con errores" a "dashboard gerencial" en un solo paso es inmanejable. Siguiendo el principio de **dividir en tareas más pequeñas**, desglosamos nuestro pipeline en 4 "piezas" fundamentales:

* **🟦 Pieza 1 (DataOps): Ingesta y Limpieza Básica:** Remover caracteres especiales o ruido de formato que no requiere IA.
* **🟩 Pieza 2 (NLP Core): Corrección Ortográfica:** Usar un modelo de lenguaje para corregir errores tipográficos (ej. "urjencia" -> "urgencia").
* **🟧 Pieza 3 (NLP Dominio): Expansión de Jerga Médica:** Traducir abreviaturas apresuradas a términos estándar (ej. "qx" -> "cirugía", "tto" -> "tratamiento").
* **🟨 Pieza 4 (Analytics): Clasificación de Negocio:** Agrupar el texto ya limpio en las categorías que los directivos necesitan evaluar (Cirugía, Ambulatorio, Derivación).

### ❓ Paso 4: Traducir en Preguntas Específicas de ML

Ya sabemos qué necesita el negocio (los directivos). Ahora debemos formular las preguntas exactas que nuestros modelos de Machine Learning van a responder. En base a nuestro desglose anterior, tenemos dos tareas predictivas principales:

1. **El Modelo Corrector (NLP):**
   * *La pregunta de negocio:* ¿Cómo arreglamos lo que escribió el médico?
   * *La pregunta de ML:* Dada una secuencia de texto ruidosa $X$, ¿cuál es la secuencia de texto más probable $Y$ en el dominio médico en español? -> **Tarea: Generación de Texto / Traducción (Sequence-to-Sequence).**

2. **El Modelo Analítico (Dashboard):**
   * *La pregunta de negocio:* ¿De qué trata este historial?
   * *La pregunta de ML:* Dado un texto limpio, ¿a qué clase discreta (Cirugía, Ambulatorio, Derivación) pertenece con mayor probabilidad? -> **Tarea: Clasificación de Texto Multiclase (Text Classification).**

## Fase 2 Priorización de casos de uso de ML
Antes de encender los servidores y gastar en cómputo, debemos evaluar si el proyecto del Hospital Y es viable y valioso. Utilizamos un framework de 4 pilares para tomar decisiones informadas:




1. **💰 Recursos y Presupuesto:** ¿Cuánto nos costará el cómputo y el talento? *Estrategia:* Aplicaremos técnicas de *low-cost ML fine-tuning* para evitar facturas astronómicas en la nube.
2. **⏱️ Plazos de Entrega (Timelines):** ¿Podemos tener un prototipo (MVP) antes del próximo cierre trimestral de los directivos?
3. **📈 Retorno de la Inversión (ROI):** ¿Cuántas horas de auditoría manual le vamos a ahorrar al equipo administrativo? Ejemplo 15 días trabajando 2 horas dirias
4. **⚠️ Riesgos Potenciales:** ¿Qué pasa si el modelo se equivoca y clasifica una cirugía como ambulatoria? ¿Cómo manejamos la privacidad de los datos de los pacientes?

## Fase 3 Desarrollo - Recopilación y Validación de Datos

Basándonos en nuestra arquitectura de diseño, procedemos a la ingesta de información. Para este proyecto, el proceso se ha optimizado cumpliendo los tres pilares de recopilación:

1. **🗄️ Identificar Fuentes:** Hemos consolidado los historiales clínicos relevantes en un archivo estructurado: `data_mlops.xlsx`.
2. **🔎 Evaluar Relevancia y Calidad:** Partimos con una ventaja operativa. El dataset proporcionado ya pasó por un proceso de curación; su calidad es alta, con precisión verificada y campos completos (sin errores de extracción).
3. **🛡️ Asegurar Viabilidad (Ética y Legalidad):** Contamos con todos los permisos organizacionales. Más importante aún, la ingesta y el procesamiento se realizan bajo estricto cumplimiento normativo **HIPAA** (Health Insurance Portability and Accountability Act), garantizando la protección y confidencialidad de la información de los pacientes de Hospital Y.

## Etapa 2: Desarrollo de modelos


## ⚙️ Fase 1: Ingeniería de Datos - Preparando el terreno para NLP

Antes de que nuestro modelo ML intente corregir ortografía o clasificar textos, debemos asegurar que la base sea sólida. Aplicaremos los 4 pasos fundamentales de la Ingeniería de Datos adaptados a texto médico:

In [ ]:
import pandas as pd

1. **🧹 Limpieza de Datos:** Manejo de notas vacías (nulos) y eliminación de caracteres extraños o "ruido" de los sistemas del hospital (ej. asteriscos, saltos de línea innecesarios).


In [ ]:
df_data = pd.read_excel("/content/drive/MyDrive/Flisol/2026/Ops ML y LLM/data/data_mlops.xlsx")

2. **🔄 Transformación de Datos:** Homogeneización de formatos. En NLP, esto implica pasar todo a minúsculas, remover acentos conflictivos y tokenización básica.


In [ ]:
df_data.dtypes

,0
correct_text,object
mistake_text,object


In [ ]:
df_data_homologate =  df_data.astype("string")

In [ ]:
df_data_homologate.dtypes

,0
correct_text,string[python]
mistake_text,string[python]


3. **⚖️ Preparación (Equilibrio de Clases):** En un hospital, tendremos miles de registros de "Medicina General" pero pocos de "Neurocirugía". Identificaremos este desbalance para evitar sesgos en el modelo clasificador final.


In [ ]:
# No tenemos clases en estos datos

4. **🧠 Ingeniería de Características (Feature Engineering):** Creación de nuevas variables útiles. Por ejemplo, la longitud de la nota clínica o banderas (flags) que indiquen si la nota contiene palabras clave de emergencia.

In [ ]:
# No tenemos columnas que nos sirvan para crear valores nuevos en estos datos

## 🧪 Fase 2: Ingeniería de Modelos (ML)
Siguiendo nuestro flujo metodológico, ahora transformamos los datos preparados en inteligencia accionable mediante tres pasos críticos:

1. **Selección de Algoritmos:** Dado que nuestro problema es de **Clasificación de Categorías** (Cirugía, Ambulatorio, etc.), elegimos **Random Forest**. Es ideal porque maneja bien datos tabulares derivados de texto (TF-IDF) y es menos propenso al sobreajuste que un árbol de decisión simple.


## Recomendación para este paso

Para este paso, recomiendo mucho lo siguiente:

### 1. Leer papers en [arXiv](https://arxiv.org/)
Primero, revisa papers en **arXiv** relacionados con tu problema.  
Esto te permitirá entender las estructuras y enfoques más recientes en el área.

- Identifica arquitecturas similares a tu caso de uso.
- Toma nota de técnicas, datasets y métricas utilizadas.

---

### 2. Construir tu propio modelo
Con base en lo aprendido, puedes:

- Diseñar tu propia arquitectura desde cero.
- Adaptar ideas de los papers a tu problema específico.

---

### 3. Usar modelos preentrenados ( [Hugging Face](https://huggingface.co/))
Otra opción muy recomendada es buscar modelos ya existentes en **Hugging Face**:

- Explora modelos ya entrenados para tareas similares.
- Evalúa su rendimiento antes de entrenar desde cero.

---

### 4. Fine-tuning (ajuste fino)
Si encuentras un buen modelo base, puedes:

- Ajustarlo a tu dataset específico.
- Mejorar su rendimiento sin entrenar desde cero.
- Ahorrar tiempo y recursos computacionales.

2. **Entrenamiento del Modelo:** El algoritmo analizará nuestros registros históricos para minimizar el error de clasificación, ajustando sus parámetros internos para reconocer patrones (ej: la relación entre "qx" y "Procedimiento Quirúrgico").


## Uso de GECTOR (versión mejorada)

Para este ejemplo, utilizaremos una librería que desarrollé a partir de un *fork* para este tipo de casos, llamada **GECTOR Improved Training**.

Puedes encontrar el repositorio aquí:

 *(https://github.com/Asuskf/gector-improved-training)*

---

## Modelo a utilizar: versión Tiny

En este caso trabajaremos con un **modelo Tiny**, que se caracteriza por ser:

- 🪶 Liviano
- ⚡ Rápido en inferencia
- 💾 Bajo consumo de memoria
- 💰 Ideal para escenarios con presupuesto limitado

---

## ¿Por qué usar modelos Tiny?

Este tipo de modelos es especialmente útil cuando:

- No se cuenta con GPUs potentes
- Se necesita despliegue en dispositivos con pocos recursos
- Se busca reducir costos de entrenamiento e inferencia
- Se prioriza velocidad sobre máxima precisión

---

## Flujo del ejemplo

En este caso práctico seguiremos este enfoque:

1. Usar la librería **GECTOR Improved Training**
2. Cargar un modelo Tiny preentrenado
3. Adaptarlo al caso de uso específico
4. Evaluar su rendimiento en un entorno limitado

---

## Nota final

Este enfoque permite experimentar con modelos de NLP avanzados sin necesidad de grandes recursos computacionales, manteniendo un balance entre eficiencia y rendimiento.

In [ ]:
%%bash
# Step 1: Clone the repository
git clone https://github.com/Asuskf/gector.git
cd gector

# Step 2: Verify that train.py exists
ls -l train.py

# Step 3: Install dependencies
pip install git+https://github.com/Asuskf/gector.git

# Step 4: Download preprocessing scripts and vocabulary
mkdir -p utils
cd utils
wget https://github.com/grammarly/gector/raw/master/utils/preprocess_data.py
wget https://raw.githubusercontent.com/grammarly/gector/master/utils/helpers.py
cd ..
mkdir -p data
cd data
wget https://github.com/grammarly/gector/raw/master/data/verb-form-vocab.txt
cd ..

-rw-r--r-- 1 root root 11436 Apr 23 17:43 train.py
  Cloning https://github.com/Asuskf/gector.git to /tmp/pip-req-build-omlc82_q
  Resolved https://github.com/Asuskf/gector.git to commit 7cf96113a3ddc631ac595b8fbb1f63ed96f05caf
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 81.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 33.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.3/153.3 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 73.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

Cloning into 'gector'...
  Running command git clone --filter=blob:none --quiet https://github.com/Asuskf/gector.git /tmp/pip-req-build-omlc82_q
--2026-04-23 17:43:40--  https://github.com/grammarly/gector/raw/master/utils/preprocess_data.py
Resolving github.com (github.com)... 140.82.121.3
Connecting to github.com (github.com)|140.82.121.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/grammarly/gector/master/utils/preprocess_data.py [following]
--2026-04-23 17:43:40--  https://raw.githubusercontent.com/grammarly/gector/master/utils/preprocess_data.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.111.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 18387 (18K) [text/plain]
Saving to: ‘preprocess_data.py’

     0K .......... .......           

In [ ]:
from google.colab import userdata
import polars as pl
import pandas as pd
from tqdm import tqdm
import re
from sklearn.model_selection import train_test_split
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer
import torch
import unicodedata
import numpy as np
from sklearn.metrics import silhouette_score
from sklearn.metrics.pairwise import euclidean_distances
from sklearn.cluster import KMeans

tqdm.pandas()

In [ ]:
df_data_homologate.head()

,correct_text,mistake_text
0,El paciente presenta dolor abdominal.,El pasiente presenta dolor adominal.
1,Se administró paracetamol de 500 mg.,Se administro parasetamol de 500 mg.
2,La presión arterial está elevada.,La precion arterial esta elevada.
3,El diagnóstico indica hipertensión crónica.,El diagnostico indica ipertension cronica.
4,Requiere cirugía de urgencia.,Requiere cirujia de urjencia.


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
df_final = df_data_homologate

In [ ]:
df_final

,correct_text,mistake_text
0,El paciente presenta dolor abdominal.,El pasiente presenta dolor adominal.
1,Se administró paracetamol de 500 mg.,Se administro parasetamol de 500 mg.
2,La presión arterial está elevada.,La precion arterial esta elevada.
3,El diagnóstico indica hipertensión crónica.,El diagnostico indica ipertension cronica.
4,Requiere cirugía de urgencia.,Requiere cirujia de urjencia.
...,...,...
95,El paciente está hidratado.,El pasiente esta hidratado.
96,Se observó mejoría clínica.,Se observo mejoria clinika.
97,El dolor es controlado.,El dolor es kontrolado.
98,El paciente fue trasladado.,El pasiente fue trasladado.


In [ ]:
df_final = df_final.drop_duplicates(subset=["correct_text", "mistake_text"])

In [ ]:
df_final.shape

(100, 2)

In [ ]:
#solo fines academicos
df_final = df_final.loc[df_final.index.repeat(110)].reset_index(drop=True)

In [ ]:
X = df_final["mistake_text"]
y = df_final["correct_text"]

In [ ]:
df = pl.DataFrame({"text": X, "label": y})

# 1) Encontrar clases con 1 solo ejemplo
counts = df.group_by("label").count()

rare_labels = (
    counts.filter(pl.col("count") == 1)
          .select("label")
          .to_series()
          .to_list()
)

# Separar clases raras
df_rare = df.filter(pl.col("label").is_in(rare_labels))
df_rest = df.filter(~pl.col("label").is_in(rare_labels))

# 1b) Proteger: si en df_rest quedan clases con solo 1 muestra
counts_rest = df_rest.group_by("label").count()

bad_labels = (
    counts_rest.filter(pl.col("count") == 1)
               .select("label")
               .to_series()
               .to_list()
)

if len(bad_labels) > 0:
    df_rare = pl.concat([df_rare, df_rest.filter(pl.col("label").is_in(bad_labels))])
    df_rest = df_rest.filter(~pl.col("label").is_in(bad_labels))

# Verificar que haya al menos 2 clases con 2+ elementos
if df_rest.select(pl.col("label").n_unique()).item() < 2:
    raise ValueError("No hay suficientes clases con 2+ ejemplos para un split estratificado.")

# Convertir a pandas para usar train_test_split
df_rest_pd = df_rest.to_pandas()

# 2) Split estratificado principal
X_train, X_temp, y_train, y_temp = train_test_split(
    df_rest_pd["text"], df_rest_pd["label"],
    test_size=0.3,
    random_state=42,
    stratify=df_rest_pd["label"]
)

# 2a) Proteger contra clases problemáticas en y_temp
counts_temp = y_temp.value_counts()
bad_temp = counts_temp[counts_temp == 1].index.tolist()

if bad_temp:
    move_to_train = df_rest_pd[df_rest_pd["label"].isin(bad_temp)]
    X_train = pd.concat([X_train, move_to_train["text"]])
    y_train = pd.concat([y_train, move_to_train["label"]])

    keep = ~X_temp.index.isin(move_to_train.index)
    X_temp = X_temp[keep]
    y_temp = y_temp[keep]

# 2b) Segundo split (validación y test)
X_val, X_unseen, y_val, y_unseen = train_test_split(
    X_temp, y_temp,
    test_size=0.5,
    random_state=42,
    stratify=y_temp
)

# 3) Construir DataFrames finales en Polars
train_df = pl.DataFrame({
    "text": pl.concat([pl.Series(X_train), df_rare["text"]]),
    "label": pl.concat([pl.Series(y_train), df_rare["label"]]),
})

val_df = pl.DataFrame({"text": X_val, "label": y_val})
unseen_df = pl.DataFrame({"text": X_unseen, "label": y_unseen})

/tmp/ipykernel_1464/3855648341.py:4: DeprecationWarning: `GroupBy.count` was renamed; use `GroupBy.len` instead
  counts = df.group_by("label").count()
/tmp/ipykernel_1464/3855648341.py:18: DeprecationWarning: `GroupBy.count` was renamed; use `GroupBy.len` instead
  counts_rest = df_rest.group_by("label").count()


In [ ]:
val_df.filter(pl.col("text").str.contains("med"))

text,label
str,str
"""Se recomienda control mediko.""","""Se recomienda control médico."""
"""El alta medica se firmara maña…","""El alta médica se firmará maña…"
"""El medico evaluo al pasiente.""","""El médico evaluó al paciente."""
"""El sertificado medico fue emit…","""El certificado médico fue emit…"
"""El medico indico reposo relati…","""El médico indicó reposo relati…"
…,…
"""El medico evaluo al pasiente.""","""El médico evaluó al paciente."""
"""El medico evaluo al pasiente.""","""El médico evaluó al paciente."""
"""Se indico segimiento medico.""","""Se indicó seguimiento médico."""


In [ ]:
list_text = []
list_label = []
for text, label in zip(train_df["text"], train_df["label"]):
  list_text.append(text)
  list_label.append(label)

In [ ]:
with open("/content/gector/data/train_source.txt", "w", encoding="utf-8") as f:
    for line in train_df["text"].to_list():
        f.write(str(line) + "\n")

In [ ]:
with open("/content/gector/data/train_target.txt", "w", encoding="utf-8") as f:
    for line in train_df["label"].to_list():
        f.write(str(line) + "\n")

In [ ]:
with open("/content/gector/data/valid_source.txt", "w", encoding="utf-8") as f:
    for line in val_df["text"].to_list():
        f.write(str(line) + "\n")

In [ ]:
with open("/content/gector/data/valid_target.txt", "w", encoding="utf-8") as f:
    for line in val_df["label"].to_list():
        f.write(str(line) + "\n")

In [ ]:
train_df.shape

(7700, 2)

In [ ]:
%%sh
cd gector

# Procesamos los datos
python utils/preprocess_data.py -s data/train_source.txt -t data/train_target.txt -o data/train_preprocessed.pt
python utils/preprocess_data.py -s data/valid_source.txt -t data/valid_target.txt -o data/valid_preprocessed.pt



The size of raw dataset is 7700
Overall extracted 7700. Original TP 7700. Original TN 0
The size of raw dataset is 1650
Overall extracted 1650. Original TP 1650. Original TN 0


7700it [00:02, 3075.93it/s]
1650it [00:00, 2353.53it/s]


3. **Optimización y Ajuste de Hiperparámetros:** No nos conformamos con el primer resultado. Utilizaremos testeo para buscar los parametros adecuados

In [ ]:
# Entrenamos el modelo
! accelerate launch gector/train.py \
--train_file gector/data/train_preprocessed.pt \
--valid_file gector/data/valid_preprocessed.pt \
--save_dir /content/drive/MyDrive/Flisol/2026/gector_test \
--model_id mrm8488/spanish-TinyBERT-betito \
--batch_size 10 \
--n_epochs 10 \
--early_stopper 8 \
--batch_size 500


The following values were not passed to `accelerate launch` and had defaults used instead:
	`--num_processes` was set to a value of `1`
	`--num_machines` was set to a value of `1`
	`--mixed_precision` was set to a value of `'no'`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
2026-04-23 17:53:39.151118: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776966819.180929    4638 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776966819.191613    4638 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776966819.220198    4638 computation_plac

## 🚀 Fase 5: Operaciones - Canalizaciones CI/CD
Para que el Hospital Y confíe en nuestras predicciones, el modelo debe pasar por un proceso automatizado, consistente y repetible:

1. **Integración Continua (CI):** * Realizamos **Pruebas Unitarias** para asegurar que el corrector ortográfico no rompa palabras clave.
   * Ejecutamos **Pruebas de Validación** con datos de prueba (Test Data) para garantizar que la precisión no baje del 85%.

2. **Despliegue Continuo (CD):** * Generamos un **Artefacto de Modelo** (un archivo `.pkl` o una imagen de contenedor).
   * Simulamos el **Despliegue a Producción** mediante una función de inferencia lista para ser consumida por el Dashboard de los directivos.

Para el CI/CD ver para [AWS](https://github.com/Asuskf/gector-improved-training/blob/refactor/gector/src/app/gector/inference_sageMaker.py) y [GCP](https://github.com/Asuskf/gector-improved-training/blob/refactor/gector/src/app/gector/inference_gcp_cloud_functions.py)


In [ ]:
from huggingface_hub import login
login()

In [ ]:
from huggingface_hub import create_repo, upload_folder
create_repo(
    repo_id="Asuskf/gector-model",
    repo_type="model"
)
upload_folder(
    repo_id="Asuskf/gector-model",
    folder_path="/content/gector_model",
    repo_type="model"
)

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...el/best/model.safetensors:  28%|##7       | 15.9MB / 57.8MB            

  ...el/last/model.safetensors:  28%|##7       | 15.9MB / 57.8MB            

No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/Asuskf/gector-model/commit/893770c34dc4636b639eba67b76cecb94608c31c', commit_message='Upload folder using huggingface_hub', commit_description='', oid='893770c34dc4636b639eba67b76cecb94608c31c', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Asuskf/gector-model', endpoint='https://huggingface.co', repo_type='model', repo_id='Asuskf/gector-model'), pr_revision=None, pr_num=None)

In [ ]:
%%writefile README.md
---
license: apache-2.0
tags:
- gector
- grammar-correction
- nlp
---

# 🧠 GECToR Model

Este modelo realiza corrección gramatical automática de texto.

## 🔧 Uso
Modelo entrenado para detectar y corregir errores gramaticales en español hambito de salud.

## 🚀 Ejemplo
Entrada: "El paciente presenta dolor abdominal."
Salida: "El pasiente presenta dolor adominal."

Overwriting README.md


## 📡 Fase 6: Monitoreo y Activación Automática
### Queda en teoría para esto necesitamos un nube en la cual podemos ocuapar
Un modelo en producción sin monitoreo es un riesgo. Para el Hospital Y, implementamos un sistema de "seguimiento y alertas" basado en cuatro pilares:

1. **⏱️ Monitoreo en Tiempo Real:** Vigilamos la **Precisión** (¿sigue clasificando bien?) y la **Latencia** (¿el dashboard carga rápido?).
| Herramienta | Tipo | Uso en el Proyecto |
|------------|------|--------------------|
| Evidently AI | Open Source / Cloud | Es la herramienta estándar para detectar Data Drift y Concept Drift. Genera reportes visuales automáticos que comparan si la jerga médica actual ha cambiado respecto a la del entrenamiento. |
| Amazon SageMaker Model Monitor | Managed Service (AWS) | Ideal si el hospital usa AWS. Monitorea automáticamente la calidad de las predicciones y envía alertas a CloudWatch si la precisión cae por debajo del 90% o la latencia sube. |
| Arize AI / Fiddler | ML Observability Platform | Plataformas especializadas en "explicabilidad". Permiten a los directivos entender por qué el modelo tomó una decisión y rastrear métricas de negocio (ROI) vinculadas al rendimiento del ML. |

2. **🚨 Sistemas de Alertas y Umbrales:** Si la precisión cae por debajo del 90% o la latencia supera los 200ms, el sistema dispara una alerta.
| Herramienta | Proveedor | Uso en el Hospital Y |
|------------|----------|------------------------|
| Amazon CloudWatch Alarms | AWS | Permite crear alarmas sobre métricas personalizadas. Si la latencia supera los 200ms, puede activar automáticamente una función Lambda para escalar servidores o notificar vía SMS/Email al equipo técnico. |
| Grafana Alerting | Multi-nube / On-premise | Ideal para visualizar umbrales. Se conecta a bases de datos de métricas (Prometheus) y permite configurar alertas visuales y sonoras en centros de control si la precisión cae del umbral definido. |
| PagerDuty | SaaS (Independiente) | Es la plataforma de respuesta a incidentes. Cuando el sistema de monitoreo detecta una degradación, PagerDuty gestiona las guardias técnicas, asegurando que el ingeniero responsable reciba la alerta de inmediato para intervenir el modelo. |

3. **🔍 Detección de Problemas (Data & Concept Drift):** Identificamos si los datos de entrada han cambiado (ej. nuevas jergas médicas) o si la relación entre las palabras y las categorías ya no es válida.
| Herramienta | Especialidad | Aplicación en el Hospital Y |
|------------|-------------|------------------------------|
| WhyLabs | Observabilidad de Datos | Funciona creando "perfiles" estadísticos de los datos. Si los doctores empiezan a usar términos de telemedicina que no estaban en el entrenamiento, WhyLabs detecta el cambio en la distribución sin necesidad de ver el contenido sensible (ideal para HIPAA). |
| Deepchecks | Validación en Producción | Ofrece un conjunto de pruebas específicas para NLP que comparan el corpus de entrenamiento con el de producción. Es excelente para identificar cuando una palabra (ej. "derivación") empieza a usarse en un contexto nuevo que el modelo no entiende. |
| Azure Machine Learning Drift Detection | Nube (Microsoft) | Si el hospital usa Azure, este servicio automatiza la comparación de datasets. Si detecta que la "distancia estadística" entre los datos de ayer y los de hoy es muy grande, marca el modelo como "degradado" y sugiere un reentrenamiento. |

4. **🔄 Activación Automática:** Ante una degradación, el sistema puede iniciar un **Pipeline de Reentrenamiento** automáticamente o notificar a los ingenieros para una intervención manual.
| Herramienta | Función Técnica | Impacto en el Hospital Y |
|------------|----------------|----------------------------|
| GitHub Actions / GitLab CI | Orquestación de Eventos | Actúa como el cerebro del flujo. Al recibir una señal de "Drift" desde el monitoreo, dispara automáticamente el script de reentrenamiento sin intervención humana. |
| Kubeflow Pipelines | Flujo de Trabajo Nativo de ML | Permite crear flujos de trabajo repetibles. Si se detecta degradación, Kubeflow vuelve a ejecutar desde la limpieza de datos hasta el despliegue de la nueva versión v2.1. |
| Slack / Microsoft Teams API | Notificación de Emergencia | Para casos críticos que requieren "intervención manual", estas herramientas envían un mensaje directo al equipo de ingenieros con el reporte del error, permitiendo una respuesta inmediata. |


## test new model

In [ ]:
from huggingface_hub import snapshot_download

model_path = snapshot_download("Asuskf/gector-model")
print(model_path)

Fetching 17 files:   0%|          | 0/17 [00:00<?, ?it/s]

best/model.safetensors:   0%|          | 0.00/57.8M [00:00<?, ?B/s]

.gitattributes: 0.00B [00:00, ?B/s]

README.md:   0%|          | 0.00/378 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/302 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

log.json: 0.00B [00:00, ?B/s]

/root/.cache/huggingface/hub/models--Asuskf--gector-model/snapshots/893770c34dc4636b639eba67b76cecb94608c31c


In [ ]:
from transformers import AutoTokenizer
from gector import GECToR, predict, load_verb_dict
import torch

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_id = '/content/gector_model/last'
model = GECToR.from_pretrained(model_id).to(device)
tokenizer = AutoTokenizer.from_pretrained(model_id)
encode, decode = load_verb_dict('/content/gector/data/verb-form-vocab.txt')
srcs = [
    'Se realizo una tomografia..',
    'El diagnostico fue tempranó.'
]
corrected = predict(
    model, tokenizer, srcs,
    encode, decode,
    keep_confidence=0.0,
    min_error_prob=0.0,
    n_iteration=5,
    batch_size=2,
)
print(corrected)

Iteratoin 0. the number of to_be_processed: 2
Iteratoin 1. the number of to_be_processed: 2
Iteratoin 2. the number of to_be_processed: 2
Iteratoin 3. the number of to_be_processed: 2
Iteratoin 4. the number of to_be_processed: 2
['Se realizó una tomografía.', 'El diagnóstico fue temprano.']
